In [1]:
from servo import Servo
import time
import threading

In [2]:
import serial.tools.list_ports

# List all available serial ports
ports = serial.tools.list_ports.comports()

print("Available serial ports:")
for port in ports:
    print(f"  {port.device}")
    print(f"    Description: {port.description}")
    print(f"    Manufacturer: {port.manufacturer}")
    print()

saved_ports = [port.device for port in ports]

Available serial ports:
  /dev/ttyACM1
    Description: USB Single Serial
    Manufacturer: None

  /dev/ttyACM0
    Description: USB Single Serial
    Manufacturer: None



In [43]:
saved_ports[0]

s=Servo(port=saved_ports[0], servo_id=6)

s.ping()

info=s.status()
print(f"Status: {info}")
print(f"Speed: {info['speed']} steps/sec")  # negative = CCW, positive = CW
print(f"Direction: {info['direction']}")
print(f"Moving: {info['moving']}")

✓ Serial comm port: /dev/ttyACM1
✓ Successfully pinged servo ID 6
✓ Model number: 777
Status: {'position': (2246, 197.40234375), 'speed': 0, 'direction': None, 'voltage': 5.5, 'temperature': 27, 'load': (0.0, 'CCW'), 'current': 0.0, 'moving': False, 'torque_enabled': False, 'mode': 0, 'errors': [], 'healthy': True}
Speed: 0 steps/sec
Direction: None
Moving: False


In [38]:
s.disable_torque()

In [5]:
from servo import Servo
import time

# Connect
s = Servo(port=saved_ports[1], servo_id=1)
s.ping()

# Check health
if not s.is_healthy():
    print(f"Warning: {s.errors()}")

# Get full status
info = s.status()
print(f"\nStarting position: {info['position'][1]:.1f}°")
print(f"Voltage: {info['voltage']:.1f}V")
print(f"Temperature: {info['temperature']}°C")

# Run a movement sequence using degrees
print("\nRunning movement sequence...")
s.move_to_degrees(0)
time.sleep(1)
s.move_to_degrees(180)
time.sleep(1)
s.move_to_degrees(360)
time.sleep(1)

# Try wheel mode
print("\nTesting wheel mode...")
s.spin(500)
time.sleep(2)
s.stop()

# Return to position mode
s.move_to(2048)
time.sleep(1)

# Final status
pos, deg = s.position()
print(f"\nFinal position: {deg:.1f}°")
print(f"Healthy: {s.is_healthy()}")

# Close bus
s.disable_torque()
s.close_bus()
print("\n✓ Done!")

✓ Serial comm port: /dev/ttyACM0
✓ Successfully pinged servo ID 1
✓ Model number: 777

Starting position: 180.0°
Voltage: 5.4V
Temperature: 27°C

Running movement sequence...



Testing wheel mode...

Final position: 179.6°
Healthy: True

✓ Done!


In [6]:
s = Servo(port=saved_ports[1], servo_id=1)
s.status()["position"][1]

179.560546875

linker arm, servo1, min 839, max=3351

3444, 773

i guess min 800, max 3400

deg: min 69, max 306

i guess min 70, max 304


In [26]:
stop_event=threading.Event()
print(s.temperature())
print(s.position())

def monitor_temp():
    while not stop_event.is_set():
        print("yey")
        # temp = s.temperature()
        # pos,_= s.position()
        # print(f"Temperature: {temp}°C, Position: {pos}")
        time.sleep(1)

thread=threading.Thread(target=monitor_temp)
thread.start()
try:
    s.move_sequence([800,3400,800,3400], tolerance =10,timeout=10)
    # healthy= s.is_healthy()
    # s.move_to(3400,speed=2000, acc=100)
    # s.ping()
    print("healthy")
except Exception as e:
    print(f"Error during move_sequence: {e}")
finally:
    stop_event.set()
    thread.join()
# s.move_sequence([800,3400,800,3400], tolerance =10,timeout=10)
# s.move_sequence([800,3400,800,3400,800,3400,800,3400,800,3400,800,3400,800,3400,800,3400,800,3400,800,3400], tolerance =10,timeout=10)
# s.status()["position"][1]

# stop_moving = True
# thread.join()

28
(3399, 298.740234375)
yey
yey
yey
yey
yey
healthy


In [22]:
targets = [1000, 2000, 3000]

for target in targets:
    s.move_to(target)

    while s.moving():
        position = s.position()
        temperature = s.temperature()

        print(position, temperature)

        time.sleep(0.1)

(2996, 263.3203125) 27
(2838, 249.43359375) 27
(2563, 225.263671875) 27
(2292, 201.4453125) 27
(2016, 177.1875) 28
(1741, 153.017578125) 27
(1468, 129.0234375) 28
(1194, 104.94140625) 27
(997, 87.626953125) 27
(1003, 88.154296875) 27
(1148, 100.8984375) 27
(1416, 124.453125) 27
(1690, 148.53515625) 28
(1957, 172.001953125) 27


In [36]:
start = time.time()
s.disable_torque()
srange=[]
while time.time() - start < 20:
    srange.append(s.position()[1])
    time.sleep(0.1)

min(srange), max(srange)

(65.0390625, 307.96875)

In [ ]:
from servo import Servo
import time
jointids=[1,2,3,4,5,6]
servopairs
# === Configuration ===
leader_port = saved_ports[0]   # Replace with your port
follower_port = saved_ports[1] # Replace with your port
duration = 20  # seconds

# === Setup ===
print("Connecting to servos...")
leader = Servo(port=leader_port)
follower = Servo(port=follower_port)

# Verify connections
leader.ping()
follower.ping()

# Synchronize starting positions
print("\nMoving to starting position...")
start_pos = 2048
leader.move_to(start_pos)
follower.move_sync(start_pos)

# Disable leader torque for manual movement
leader.disable_torque()

# === Teleoperation Loop ===
print(f"\nTeleoperation active for {duration} seconds.")
print("Move the leader servo by hand - the follower will mirror!")

start = time.time()

while time.time() - start < duration:
    # Read leader position and speed
    info = leader.status()
    pos = info['position'][0]
    speed = abs(info['speed'])

    # Command follower
    follower.move_to(pos, speed=speed)

# === Cleanup ===
print("Teleoperation ended.")
leader.disable_torque()
follower.disable_torque()
leader.close_bus()
follower.close_bus()
print("Both servos shut down.")

Connecting to servos...
✓ Serial comm port: /dev/ttyACM1
✓ Successfully pinged servo ID 1
✓ Model number: 777
✓ Serial comm port: /dev/ttyACM0
✓ Successfully pinged servo ID 1
✓ Model number: 777

Moving to starting position...

Teleoperation active for 20 seconds.
Move the leader servo by hand - the follower will mirror!
Teleoperation ended.
Both servos shut down.
